# Reviewer revision analyses: HCFIM EEG study

## tl;dr

This notebook consolidates the analyses added during peer review of the HCFIM EEG manuscript. It reproduces the reported nested-CV comparisons, reviewer-requested PR-AUC and confusion-matrix summaries, prevalence-dependent PPV, H/C/FIM redundancy checks, epoch-count diagnostics, and the corrected regional-versus-lateralized comparison figure.

Key reproduced findings:

- Direct AD-versus-FTD discrimination was weak: ROC AUC = **0.471** (PSD), **0.335** (HCFIM), and **0.324** (combined).
- The three-class combined model reached macro one-vs-rest ROC AUC = **0.662**, balanced accuracy = **0.509**, and macro F1 = **0.482**.
- H/C/FIM were strongly redundant in the two highlighted configurations: PC1 explained **96.6%** and **84.3%** of standardized variance.
- Retained epochs varied by diagnosis and subject: AD **60–255**, control **148–196**, and FTD **94–201** epochs.

All model-selection and preprocessing steps are fitted inside the training folds. The notebook reads only versioned project outputs and writes reviewer-specific derivatives to `results/reviewer_revision/`.

## Context & Methods

The purpose is auditability of the revision, not a new exploratory analysis. Binary results use out-of-fold (OOF) predictions from the released nested cross-validation workflow. Reviewer-specific differential models use five outer folds and three inner folds, with median imputation, standardization, univariate feature selection, and Extra Trees hyperparameter selection performed inside each training fold.

### Key assumptions

- `results/subject_features_full.csv` is the subject-level matrix used by the manuscript.
- Existing regional and lateralized OOF prediction files are authoritative for the already-released control-versus-disease models.
- Bootstrap intervals resample subjects with replacement and are descriptive 95% percentile intervals.
- PPV at 5% and 10% prevalence is a transport calculation from observed sensitivity and specificity; it is not an externally validated screening estimate.
- PCA quantifies redundancy among the three HCFIM coordinates in the two configurations highlighted in the manuscript; it does not test diagnostic separation.


In [ ]:
from __future__ import annotations

import json
import os
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

SEED = 17536
BOOTSTRAP_SEED = 20260818
N_BOOTSTRAP = 4000

def resolve_project_root() -> Path:
    candidates = []
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "results" / "subject_features_full.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run from the repository/notebooks directory "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS = PROJECT_ROOT / "results"
OUT = RESULTS / "reviewer_revision"
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9.0, 5.2),
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})
COLORS = {"regional": "#214d7d", "lateralized": "#69b3c2", "psd": "#214d7d", "hcfim": "#d58a2a", "psd_plus_hcfim": "#6f8f3d"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} | pandas: {pd.__version__}")
print(f"Reviewer outputs: {OUT}")

## Data

### 1. Validate and load subject-level inputs

The checks below fail early if a required released artifact is missing. The 3.3 GB epoch-level file is deliberately deferred until the epoch-count section.

In [ ]:
required_files = [
    RESULTS / "subject_features_full.csv",
    RESULTS / "classification_results_all_schemas_with_bootstrap_ci.csv",
    RESULTS / "epoch_features_full.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, f"Missing required files: {missing}"

subject_features = pd.read_csv(RESULTS / "subject_features_full.csv")
all_metrics = pd.read_csv(RESULTS / "classification_results_all_schemas_with_bootstrap_ci.csv")

diagnosis_counts = subject_features["diagnosis"].value_counts().rename_axis("diagnosis").rename("subjects")
assert diagnosis_counts.to_dict() == {
    "alzheimer": 36,
    "control": 29,
    "frontotemporal_dementia": 23,
}
assert subject_features["participant_id"].nunique() == 88

display(diagnosis_counts.to_frame())
print(f"Subject-level matrix: {subject_features.shape[0]} rows × {subject_features.shape[1]} columns")

## Results

### 2. Reviewer-requested metrics for released control-versus-disease models

ROC AUC and PR-AUC are recomputed directly from OOF predictions. Balanced accuracy, F1, sensitivity, specificity, and confusion counts are reconciled with the released summary table. PPV is then projected to 5% and 10% prevalence.

In [ ]:
def bootstrap_binary(y: np.ndarray, probability: np.ndarray, *, seed: int = BOOTSTRAP_SEED, n: int = N_BOOTSTRAP) -> dict:
    rng = np.random.default_rng(seed)
    aucs, aps = [], []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if np.unique(y[idx]).size < 2:
            continue
        aucs.append(roc_auc_score(y[idx], probability[idx]))
        aps.append(average_precision_score(y[idx], probability[idx]))
    return {
        "roc_auc": roc_auc_score(y, probability),
        "roc_auc_low": np.percentile(aucs, 2.5),
        "roc_auc_high": np.percentile(aucs, 97.5),
        "pr_auc": average_precision_score(y, probability),
        "pr_auc_low": np.percentile(aps, 2.5),
        "pr_auc_high": np.percentile(aps, 97.5),
    }

def projected_ppv(sensitivity: float, specificity: float, prevalence: float) -> float:
    numerator = sensitivity * prevalence
    denominator = numerator + (1.0 - specificity) * (1.0 - prevalence)
    return numerator / denominator

candidate_models = [
    ("regional_v1", "control_vs_alzheimer", "psd", "extra_trees"),
    ("regional_v1", "control_vs_alzheimer", "hcfim", "extra_trees"),
    ("regional_v1", "control_vs_alzheimer", "psd_plus_hcfim", "extra_trees"),
    ("lateralized_v1", "control_vs_alzheimer", "psd_plus_hcfim_plus_lateral_deltas", "extra_trees"),
    ("regional_v1", "control_vs_frontotemporal_dementia", "psd", "extra_trees"),
    ("regional_v1", "control_vs_frontotemporal_dementia", "hcfim", "extra_trees"),
    ("regional_v1", "control_vs_frontotemporal_dementia", "psd_plus_hcfim", "extra_trees"),
    ("regional_v1", "control_vs_frontotemporal_dementia", "hcfim", "mlp_small"),
    ("lateralized_v1", "control_vs_frontotemporal_dementia", "psd_plus_hcfim", "logistic_elasticnet"),
]

reviewer_rows = []
for schema, comparison, family, model in candidate_models:
    prefix = "oof_predictions_" if schema == "regional_v1" else "oof_predictions_lateralized_v1_"
    prediction_path = RESULTS / f"{prefix}{comparison}_{family}_{model}.csv"
    predictions = pd.read_csv(prediction_path)
    y = predictions["y_true"].to_numpy(int)
    probability = predictions["y_prob"].to_numpy(float)
    row = bootstrap_binary(y, probability)
    released = all_metrics.loc[
        (all_metrics["schema"] == schema)
        & (all_metrics["comparison"] == comparison)
        & (all_metrics["feature_family"] == family)
        & (all_metrics["model"] == model)
    ]
    assert len(released) == 1
    released = released.iloc[0]
    row.update({
        "schema": schema,
        "comparison": comparison,
        "feature_family": family,
        "model": model,
        "n_subjects": int(released["n_subjects"]),
        "balanced_accuracy": float(released["balanced_accuracy"]),
        "balanced_accuracy_low": float(released["balanced_accuracy_ci_low"]),
        "balanced_accuracy_high": float(released["balanced_accuracy_ci_high"]),
        "f1": float(released["f1"]),
        "f1_low": float(released["f1_ci_low"]),
        "f1_high": float(released["f1_ci_high"]),
        "sensitivity": float(released["sensitivity"]),
        "specificity": float(released["specificity"]),
        "tn": int(released["tn"]), "fp": int(released["fp"]),
        "fn": int(released["fn"]), "tp": int(released["tp"]),
        "ppv_prevalence_05": projected_ppv(float(released["sensitivity"]), float(released["specificity"]), 0.05),
        "ppv_prevalence_10": projected_ppv(float(released["sensitivity"]), float(released["specificity"]), 0.10),
    })
    reviewer_rows.append(row)

reviewer_metrics = pd.DataFrame(reviewer_rows)
reviewer_metrics.to_csv(OUT / "reviewer_classification_metrics.csv", index=False)

display_columns = [
    "schema", "comparison", "feature_family", "model", "n_subjects",
    "roc_auc", "roc_auc_low", "roc_auc_high", "pr_auc",
    "balanced_accuracy", "sensitivity", "specificity",
    "ppv_prevalence_05", "ppv_prevalence_10",
]
display(reviewer_metrics[display_columns].style.format(precision=3))

In [ ]:
plot_data = reviewer_metrics.copy()
plot_data["task"] = plot_data["comparison"].map({
    "control_vs_alzheimer": "Control vs AD",
    "control_vs_frontotemporal_dementia": "Control vs FTD",
})
family_labels = {
    "psd": "PSD",
    "hcfim": "HCFIM",
    "psd_plus_hcfim": "PSD + HCFIM",
    "psd_plus_hcfim_plus_lateral_deltas": "PSD + HCFIM + lateral deltas",
}
model_labels = {
    "extra_trees": "Extra Trees",
    "mlp_small": "MLP",
    "logistic_elasticnet": "Elastic-net logistic",
}
plot_data["label"] = [
    f"{'Regional' if schema == 'regional_v1' else 'Lateralized'} · {family_labels[family]} · {model_labels[model]}"
    for schema, family, model in zip(plot_data["schema"], plot_data["feature_family"], plot_data["model"])
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), sharex=False)
for ax, (task, group) in zip(axes, plot_data.groupby("task", sort=False)):
    ordered = group.sort_values("roc_auc")
    colors = [COLORS["lateralized"] if s == "lateralized_v1" else COLORS["regional"] for s in ordered["schema"]]
    ax.barh(ordered["label"], ordered["roc_auc"], color=colors)
    ax.errorbar(
        ordered["roc_auc"], np.arange(len(ordered)),
        xerr=[ordered["roc_auc"] - ordered["roc_auc_low"], ordered["roc_auc_high"] - ordered["roc_auc"]],
        fmt="none", ecolor="#333333", capsize=3, linewidth=1.1,
    )
    for y_pos, value in enumerate(ordered["roc_auc"]):
        ax.text(value - 0.012, y_pos, f"{value:.3f}", ha="right", va="center", fontsize=8, color="white", fontweight="bold")
    ax.axvline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.set_xlim(0.45, 1.0)
    ax.set_title(f"{task}: OOF ROC AUC (95% bootstrap CI)")
    ax.set_xlabel("ROC AUC")
    ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES / "reviewer_candidate_model_auc.png", bbox_inches="tight", facecolor="white")
plt.show()

print("Interpretation: combined regional features improved the released control-versus-disease models, but projected PPV remains modest at screening prevalences.")

### 3. Direct AD-versus-FTD and three-class nested-CV models

These analyses address whether the feature space differentiates the two dementia syndromes directly and whether a single model can separate all three diagnostic classes. The exact pipeline and random seeds are fixed below.

In [ ]:
def feature_columns(frame: pd.DataFrame, family: str) -> list[str]:
    psd = [column for column in frame if column.startswith("psd__")]
    hcfim = [column for column in frame if column.startswith("hcfim__")]
    mapping = {"psd": psd, "hcfim": hcfim, "psd_plus_hcfim": psd + hcfim}
    return mapping[family]

def nested_oof(frame: pd.DataFrame, groups: list[str], family: str) -> tuple[dict, pd.DataFrame]:
    subset = frame.loc[frame["diagnosis"].isin(groups)].copy()
    columns = feature_columns(subset, family)
    label_map = {group: index for index, group in enumerate(groups)}
    y = subset["diagnosis"].map(label_map).to_numpy(int)
    X = subset[columns].to_numpy(float)
    n_classes = len(groups)
    probabilities = np.full((len(y), n_classes), np.nan)
    predictions = np.full(len(y), -1)

    outer_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(30, X.shape[1]))),
            ("model", ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            )),
        ])
        inner_cv = StratifiedKFold(3, shuffle=True, random_state=SEED + fold)
        scoring = "roc_auc" if n_classes == 2 else "roc_auc_ovr_weighted"
        search = GridSearchCV(
            pipeline,
            {"model__max_depth": [2, 3, None]},
            scoring=scoring,
            cv=inner_cv,
            n_jobs=1,
        )
        search.fit(X[train_idx], y[train_idx])
        fold_probability = search.best_estimator_.predict_proba(X[test_idx])
        probabilities[test_idx, :] = fold_probability
        predictions[test_idx] = np.argmax(fold_probability, axis=1)

    metrics = {
        "groups": " | ".join(groups),
        "feature_family": family,
        "n_subjects": len(y),
        "n_features": len(columns),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "f1_macro": f1_score(y, predictions, average="macro"),
        "confusion_matrix": json.dumps(confusion_matrix(y, predictions).tolist()),
    }
    if n_classes == 2:
        metrics["roc_auc"] = roc_auc_score(y, probabilities[:, 1])
    else:
        metrics["roc_auc_macro_ovr"] = roc_auc_score(
            label_binarize(y, classes=np.arange(n_classes)),
            probabilities,
            multi_class="ovr",
            average="macro",
        )

    oof = pd.DataFrame({
        "participant_id": subset["participant_id"].to_numpy(),
        "diagnosis": subset["diagnosis"].to_numpy(),
        "y_true": y,
        "y_pred": predictions,
    })
    for class_index, group in enumerate(groups):
        oof[f"p_{group}"] = probabilities[:, class_index]
    return metrics, oof

differential_runs = []
for family in ["psd", "hcfim", "psd_plus_hcfim"]:
    differential_runs.append((["alzheimer", "frontotemporal_dementia"], family))
differential_runs.append((["control", "alzheimer", "frontotemporal_dementia"], "psd_plus_hcfim"))

differential_rows = []
differential_oof = {}
for groups, family in differential_runs:
    metrics, oof = nested_oof(subject_features, groups, family)
    key = f"{'_vs_'.join(groups)}_{family}_extra_trees"
    differential_rows.append(metrics)
    differential_oof[key] = oof
    oof.to_csv(OUT / f"oof_{key}.csv", index=False)

differential_metrics = pd.DataFrame(differential_rows)
differential_metrics.to_csv(OUT / "differential_classification_metrics.csv", index=False)
display(differential_metrics.drop(columns="confusion_matrix").style.format(precision=3))

In [ ]:
expected = {
    "psd": {"roc_auc": 0.4710144927536232, "balanced_accuracy": 0.4716183574879227},
    "hcfim": {"roc_auc": 0.33454106280193235, "balanced_accuracy": 0.42814009661835745},
    "psd_plus_hcfim": {"roc_auc": 0.32367149758454106, "balanced_accuracy": 0.40640096618357485},
}
binary_rows = differential_metrics[differential_metrics["groups"] == "alzheimer | frontotemporal_dementia"].set_index("feature_family")
for family, values in expected.items():
    for metric, target in values.items():
        assert np.isclose(binary_rows.loc[family, metric], target, atol=1e-12), (family, metric)

three_class = differential_metrics[differential_metrics["groups"].str.startswith("control |")].iloc[0]
assert np.isclose(three_class["roc_auc_macro_ovr"], 0.6615635522091192, atol=1e-12)
assert np.isclose(three_class["balanced_accuracy"], 0.5093147870509189, atol=1e-12)
assert np.isclose(three_class["f1_macro"], 0.4821734464930251, atol=1e-12)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))

ordered = binary_rows.loc[["psd", "hcfim", "psd_plus_hcfim"]]
labels = ["PSD", "HCFIM", "PSD + HCFIM"]
bars = axes[0].bar(labels, ordered["roc_auc"], color=[COLORS[k] for k in ordered.index])
axes[0].axhline(0.5, color="#555555", linestyle="--", linewidth=1, label="Chance")
axes[0].bar_label(bars, fmt="%.3f", padding=3)
axes[0].set_ylim(0, 0.7)
axes[0].set_ylabel("OOF ROC AUC")
axes[0].set_title("Direct AD vs FTD")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

three_key = "control_vs_alzheimer_vs_frontotemporal_dementia_psd_plus_hcfim_extra_trees"
three_oof = differential_oof[three_key]
ConfusionMatrixDisplay.from_predictions(
    three_oof["y_true"], three_oof["y_pred"],
    display_labels=["Control", "AD", "FTD"],
    cmap="Blues", colorbar=False, ax=axes[1],
)
axes[1].set_title("Three-class OOF confusion matrix")
fig.tight_layout()
fig.savefig(FIGURES / "differential_and_multiclass_performance.png", bbox_inches="tight", facecolor="white")
plt.show()

print("Interpretation: the direct disease-versus-disease models did not exceed chance ROC AUC; the three-class model was only moderately discriminative and confused FTD most often.")

### 4. H/C/FIM coordinate redundancy

PCA is fitted to standardized H, C, and FIM values for the two regional configurations highlighted in the manuscript. Pairwise correlations and explained variance are reported together so that the redundancy claim is directly inspectable.

In [ ]:
pca_configs = [
    ("AD temporal alpha", "control_vs_alzheimer", ["control", "alzheimer"], "temporal", "alpha"),
    ("FTD temporal broadband", "control_vs_frontotemporal_dementia", ["control", "frontotemporal_dementia"], "temporal", "broadband"),
]

pca_rows = []
for label, comparison, groups, region, band in pca_configs:
    prefix = f"hcfim__ordinal_shannon_m4_t1__euclidean__{region}__{band}"
    columns = [f"{prefix}__{coordinate}__mean" for coordinate in ["H", "C", "FIM"]]
    subset = subject_features.loc[subject_features["diagnosis"].isin(groups), columns].dropna()
    standardized = StandardScaler().fit_transform(subset)
    fit = PCA(n_components=3).fit(standardized)
    correlations = subset.corr()
    pca_rows.append({
        "configuration": label,
        "comparison": comparison,
        "n": len(subset),
        "pc1_variance": fit.explained_variance_ratio_[0],
        "pc2_variance": fit.explained_variance_ratio_[1],
        "pc3_variance": fit.explained_variance_ratio_[2],
        "pc1_pc2_cumulative": fit.explained_variance_ratio_[:2].sum(),
        "corr_H_C": correlations.iloc[0, 1],
        "corr_H_FIM": correlations.iloc[0, 2],
        "corr_C_FIM": correlations.iloc[1, 2],
    })

pca_results = pd.DataFrame(pca_rows)
pca_results.to_csv(OUT / "hcfim_coordinate_pca.csv", index=False)
assert np.isclose(pca_results.loc[0, "pc1_variance"], 0.9664622386614715, atol=1e-12)
assert np.isclose(pca_results.loc[1, "pc1_variance"], 0.8425806077671439, atol=1e-12)
display(pca_results.style.format(precision=4))

variance_plot = pca_results.set_index("configuration")[["pc1_variance", "pc2_variance", "pc3_variance"]]
ax = variance_plot.plot.bar(stacked=True, color=["#214d7d", "#d58a2a", "#d9dde2"], figsize=(8.5, 4.5))
ax.set_ylabel("Explained variance ratio")
ax.set_xlabel("")
ax.set_title("Redundancy among standardized H, C, and FIM coordinates")
ax.legend(["PC1", "PC2", "PC3"], frameon=False, ncol=3)
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.2)
for index, value in enumerate(pca_results["pc1_variance"]):
    ax.text(index, value / 2, f"PC1 {value:.1%}", ha="center", va="center", color="white", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "hcfim_coordinate_redundancy.png", bbox_inches="tight", facecolor="white")
plt.show()

print("Interpretation: the three coordinates are not independent in these configurations; most standardized variance lies on one principal axis.")

### 5. Retained epoch counts

The epoch table is read with only three columns. Counts are based on unique `(participant_id, epoch_id)` pairs, preventing feature rows or repeated configurations from inflating the denominator.

In [ ]:
epoch_source = RESULTS / "epoch_features_full.csv"
epoch_index = pd.read_csv(epoch_source, usecols=["participant_id", "diagnosis", "epoch_id"])
epoch_counts = (
    epoch_index.drop_duplicates(["participant_id", "epoch_id"])
    .groupby(["participant_id", "diagnosis"])
    .size()
    .rename("epoch_count")
    .reset_index()
)
del epoch_index

epoch_summary = (
    epoch_counts.groupby("diagnosis")["epoch_count"]
    .agg(["count", "min", "median", "max", "mean", "std"])
    .reset_index()
)
epoch_counts.to_csv(OUT / "epoch_counts_by_subject.csv", index=False)
epoch_summary.to_csv(OUT / "epoch_count_summary.csv", index=False)

expected_ranges = {
    "alzheimer": (60, 255),
    "control": (148, 196),
    "frontotemporal_dementia": (94, 201),
}
for diagnosis, (minimum, maximum) in expected_ranges.items():
    row = epoch_summary.set_index("diagnosis").loc[diagnosis]
    assert int(row["min"]) == minimum and int(row["max"]) == maximum
display(epoch_summary.style.format({"median": "{:.1f}", "mean": "{:.1f}", "std": "{:.1f}"}))

order = ["control", "alzheimer", "frontotemporal_dementia"]
labels = ["Control", "AD", "FTD"]
series = [epoch_counts.loc[epoch_counts["diagnosis"] == diagnosis, "epoch_count"] for diagnosis in order]
fig, ax = plt.subplots(figsize=(7.4, 4.6))
box = ax.boxplot(series, tick_labels=labels, patch_artist=True, showmeans=True)
for patch, color in zip(box["boxes"], ["#214d7d", "#d58a2a", "#6f8f3d"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for index, values in enumerate(series, start=1):
    jitter = np.linspace(-0.08, 0.08, len(values))
    ax.scatter(index + jitter, values, s=13, facecolors="white", edgecolors="#333333", linewidths=0.5, alpha=0.8)
ax.set_ylabel("Retained epochs per subject")
ax.set_title("Unequal retained-epoch counts across subjects and diagnoses")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES / "retained_epoch_counts.png", bbox_inches="tight", facecolor="white")
plt.show()

print("Interpretation: controls had the narrowest epoch-count range; AD and FTD showed wider variation, supporting explicit reporting and sensitivity analysis.")

### 6. Corrected regional-versus-lateralized comparison (Figure 7)

The comparison is derived from the reviewer metric table rather than hard-coded. For each task, it uses the manuscript's selected regional and lateralized candidate model and its subject-bootstrap ROC AUC interval.

In [ ]:
figure7_specs = [
    ("Control vs AD", "regional_v1", "control_vs_alzheimer", "psd_plus_hcfim", "extra_trees"),
    ("Control vs AD", "lateralized_v1", "control_vs_alzheimer", "psd_plus_hcfim_plus_lateral_deltas", "extra_trees"),
    ("Control vs FTD", "regional_v1", "control_vs_frontotemporal_dementia", "hcfim", "mlp_small"),
    ("Control vs FTD", "lateralized_v1", "control_vs_frontotemporal_dementia", "psd_plus_hcfim", "logistic_elasticnet"),
]

figure7_rows = []
for task, schema, comparison, family, model in figure7_specs:
    match = reviewer_metrics.loc[
        (reviewer_metrics["schema"] == schema)
        & (reviewer_metrics["comparison"] == comparison)
        & (reviewer_metrics["feature_family"] == family)
        & (reviewer_metrics["model"] == model)
    ]
    assert len(match) == 1
    row = match.iloc[0]
    figure7_rows.append({
        "task": task,
        "schema": "Regional" if schema == "regional_v1" else "Lateralized",
        "roc_auc": row["roc_auc"],
        "ci_low": row["roc_auc_low"],
        "ci_high": row["roc_auc_high"],
    })
figure7_data = pd.DataFrame(figure7_rows)
figure7_data.to_csv(OUT / "figure7_source_data.csv", index=False)
display(figure7_data.style.format(precision=3))

tasks = ["Control vs AD", "Control vs FTD"]
regional = figure7_data[figure7_data["schema"] == "Regional"].set_index("task").loc[tasks]
lateralized = figure7_data[figure7_data["schema"] == "Lateralized"].set_index("task").loc[tasks]
x = np.arange(len(tasks))
width = 0.34
fig, ax = plt.subplots(figsize=(8.6, 4.5))
bars_regional = ax.bar(x - width / 2, regional["roc_auc"], width, label="Regional", color=COLORS["regional"])
bars_lateralized = ax.bar(x + width / 2, lateralized["roc_auc"], width, label="Lateralized", color=COLORS["lateralized"])
ax.errorbar(
    x - width / 2, regional["roc_auc"],
    yerr=[regional["roc_auc"] - regional["ci_low"], regional["ci_high"] - regional["roc_auc"]],
    fmt="none", ecolor="#222222", capsize=4, linewidth=1.4,
)
ax.errorbar(
    x + width / 2, lateralized["roc_auc"],
    yerr=[lateralized["roc_auc"] - lateralized["ci_low"], lateralized["ci_high"] - lateralized["roc_auc"]],
    fmt="none", ecolor="#222222", capsize=4, linewidth=1.4,
)
ax.bar_label(bars_regional, fmt="%.3f", padding=4, fontsize=9)
ax.bar_label(bars_lateralized, fmt="%.3f", padding=4, fontsize=9)
ax.set_ylabel("ROC AUC")
ax.set_xticks(x, tasks)
ax.set_ylim(0.5, 1.0)
ax.legend(frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.22)
fig.tight_layout()
figure7_path = FIGURES / "figure7_regional_vs_lateralized.png"
fig.savefig(figure7_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved corrected Figure 7: {figure7_path}")

## Takeaways

- The reviewer analyses reproduce the values reported in the revised manuscript, with explicit assertions against the reported direct differential and three-class results.
- HCFIM added value for the released control-versus-disease models, especially when combined with PSD, but direct AD-versus-FTD discrimination remained poor in this sample.
- The H/C/FIM coordinates are highly redundant in the two highlighted configurations; the three-dimensional representation should therefore be interpreted as a geometric descriptor rather than three independent biomarkers.
- Unequal retained-epoch counts are documented and remain a limitation. A fixed-duration sensitivity analysis is separate confirmatory work and is not claimed here.
- The screening PPV calculations depend strongly on assumed prevalence and do not replace external validation or calibration in a target clinical population.


In [ ]:
manifest = {
    "project_root": str(PROJECT_ROOT),
    "seed": SEED,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_replicates": N_BOOTSTRAP,
    "n_subjects": int(subject_features["participant_id"].nunique()),
    "outputs": sorted(str(path.relative_to(PROJECT_ROOT)) for path in OUT.rglob("*") if path.is_file()),
}
(OUT / "notebook_05_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Validation status: all numerical assertions passed.")
print(f"Created {len(manifest['outputs'])} reviewer-revision output files.")
print(json.dumps(manifest, indent=2))